# L01 · Turning Text into Nested Data

*Build-Up Series · Lab 1 of 15 · standard library only · saves the module file `grid_pieces.py`*

**How to work through this notebook.** Do not just run the cells. Type each code cell yourself, in your own notebook or in one growing file, run it, and compare what you see with the output printed under it. Every code cell is small on purpose and every one is followed by an explanation of what it did and why. Boxes marked **Try it** are guided experiments: copy your file, make the named change, look at what happens, then go back to the original.

By the end you will be able to take a messy block of text, turn it into a nested structure of dicts, lists and tuples, and read any piece of that structure with a one-line path. That skill is what makes a lot of "complicated" code stop looking complicated, because most complicated code is just nested data being passed between small functions.

## 1 · The problem

A building-design tool exports the floor plan of a small bakery as one block of text. Your program has to answer questions such as:

* Is the square at row 3, column 5 a wall?
* Where are the doors?
* From this square, which squares can I step to?

Text cannot answer those questions. To a program, the floor plan is one long string with invisible line-break characters in it, and finding "row 3, column 5" by hand means counting characters across line breaks, which is exactly where off-by-one mistakes breed.

**Analogy.** A printed seating chart on a wall is easy for a person: you glance at it. A program cannot glance. It needs the seating chart turned into a database where "table 3, seat 5" is a lookup. This lab is the translation from the printed chart to the database.

## 2 · Tiny version (check it with a pencil)

We start with a map so small that you can verify every result on paper.

**Step 1: what is this text, really?**

In [1]:
# A tiny floor plan: '#' is wall, '.' is floor.
tiny_map_text = """####
#..#
####"""

print(type(tiny_map_text))   # what kind of object is it?
print(len(tiny_map_text))    # how many characters does it hold?
print(repr(tiny_map_text))   # repr shows the characters a normal print hides

<class 'str'>
14
'####\n#..#\n####'


**What happened.** `type` says this is a `str`, a *string*: an ordered sequence of characters. `len` says it holds 14 characters. Check with a pencil: three lines of 4 characters is 12, plus 2 line-break characters between the lines is 14. `repr` makes the hidden ones visible: `\n` is the single character that means "start a new line". The line break is not magic formatting, it is a character that takes up space like any other.

**Step 2: cut the text into one string per line.**

In [2]:
# splitlines() cuts the string at every line break and returns a list of strings.
tiny_rows_as_text = tiny_map_text.splitlines()

print(tiny_rows_as_text)
print(len(tiny_rows_as_text))   # three lines, so three strings

['####', '#..#', '####']
3


**What happened.** `splitlines()` returned a *list* (an ordered, numbered collection) of three strings and threw the line-break characters away, since they were only separators.

**Analogy.** Picture a long paper tape with the whole map printed on it. `splitlines()` cuts the tape at every line break and gives you three strips, one per row. Each strip is still a string.

**Step 3: ask for one square with two lookups.**

In [3]:
row_index = 1      # counting starts at 0, so 1 means "the second row"
column_index = 2   # and 2 means "the third character across"

one_row = tiny_rows_as_text[row_index]     # first lookup: pick the strip
one_character = one_row[column_index]      # second lookup: pick the character on it

print(one_row, one_character)
print(tiny_rows_as_text[1][2])             # the same thing as one expression

#..# .
.


**What happened.** `tiny_rows_as_text[1]` gave back the second strip, `'#..#'`, and `[2]` on that strip gave back its third character, `.`. Writing `tiny_rows_as_text[1][2]` just does both lookups left to right: the first brackets are evaluated first, and whatever they return gets the second brackets.

**Two things to memorise now.** First, the order is *(row, column)*: row counts down the page, column counts across it. That is the opposite of the *(x, y)* order from school graphs. Second, counting starts at 0.

**Analogy.** A filing cabinet: the first number picks the drawer, the second picks the folder inside the drawer.

**Step 4: strings cannot be edited, lists can.**

In [4]:
# Try to change one character of a string row.
try:
    one_row[2] = "#"
except TypeError as problem:
    print("TypeError:", problem)

# A list comprehension turns each row string into a list of one-character strings.
# Read it aloud as: "for each row_text in tiny_rows_as_text, make list(row_text)".
tiny_grid = [list(row_text) for row_text in tiny_rows_as_text]

# The same thing written as an ordinary loop, so you can see there is no magic:
#   tiny_grid = []
#   for row_text in tiny_rows_as_text:
#       tiny_grid.append(list(row_text))

tiny_grid[1][2] = "#"   # a list of characters can be edited in place
print(tiny_grid)

TypeError: 'str' object does not support item assignment
[['#', '#', '#', '#'], ['#', '.', '#', '#'], ['#', '#', '#', '#']]


**What happened.** Python refuses to change a character inside a string: strings are *immutable* (they never change after they are created). A list is *mutable*, so after `list(row_text)` turns `'#..#'` into `['#', '.', '.', '#']`, we could overwrite a single character. The list of lists is called a **grid**, and `tiny_grid[1][2]` is the same two-step lookup as before, now usable for both reading and writing.

Notice that we changed `tiny_grid` but the original `tiny_map_text` is untouched: `list(...)` made copies of the characters.

**Pencil check.** Row 1 originally read `# . . #`. We overwrote column 2 with `#`, so row 1 should now read `# . # #`, which is exactly what the printed grid shows.

## 3 · Grow it: the same idea on a messy, realistic file

Real files are never as tidy as the tiny map. Here is what the design tool actually exports. Look for the messes before reading on:

* a blank line at the start and at the end of the file,
* a **header** (`name:` and `legend:` lines) before the map itself,
* trailing spaces after some rows (the tool did not clean up),
* rows of **different lengths**, because the bottom-right corner of the building is cut away and the tool leaves that part blank.

In [5]:
bakery_file_text = """
name: Corner Bakery
legend: # wall | . floor | D door | S start | E exit

##########
#S...#...#~~~
#.##.#.#.#
#.#..D.#E#
#.#.####.#
#...D...#
#########
"""

# Text editors love to delete trailing spaces, so the three ~ characters above stand
# in for three real trailing spaces, which we put back here:
bakery_file_text = bakery_file_text.replace("~~~", "   ")

print(repr(bakery_file_text[:60]))   # the first 60 characters, hidden ones visible

'\nname: Corner Bakery\nlegend: # wall | . floor | D door | S s'


**Step 5: split the file into its header and its map.**

In [6]:
# partition(separator) cuts the string at the FIRST occurrence of the separator
# and returns a tuple of three pieces: (before, the separator itself, after).
cleaned_file_text = bakery_file_text.strip("\n")   # drop blank lines at both ends
header_text, _, map_text = cleaned_file_text.partition("\n\n")   # a blank line separates header and map

print("HEADER:")
print(header_text)
print()
print("MAP:")
print(map_text)

HEADER:
name: Corner Bakery
legend: # wall | . floor | D door | S start | E exit

MAP:
##########
#S...#...#   
#.##.#.#.#
#.#..D.#E#
#.#.####.#
#...D...#
#########


**What happened.** A blank line is the two characters `\n\n` in a row, so `partition("\n\n")` splits the file there. It returns a **tuple** of three things; we keep the first and the last. The middle one is the separator itself, which we do not need, so we name it `_`, the usual Python name for "a value I am throwing away".

A *tuple* is like a list that is fixed in size and cannot be changed. It is the right container for "a small, fixed number of related values", like these three pieces. You will use tuples constantly in this series.

**Step 6: turn the header lines into a dict, and the legend into a second dict.**

In [7]:
header = {}
for line in header_text.splitlines():
    key, _, value = line.partition(":")        # split at the FIRST colon only
    header[key.strip()] = value.strip()        # strip() removes spaces at both ends

print(header)

# The legend text looks like "# wall | . floor | D door | ...".
# Cut at '|', then split each entry into its symbol and its meaning.
legend = {}
for entry in header["legend"].split("|"):
    symbol, _, meaning = entry.strip().partition(" ")
    legend[symbol] = meaning.strip()

print(legend)

{'name': 'Corner Bakery', 'legend': '# wall | . floor | D door | S start | E exit'}
{'#': 'wall', '.': 'floor', 'D': 'door', 'S': 'start', 'E': 'exit'}


**What happened.** A **dict** (dictionary) is a lookup by name: `header["legend"]` finds the value filed under the key `"legend"`. Two dicts came out. `header` maps header names to text. `legend` maps each map symbol to its meaning, for example `'#'` to `'wall'`. Reading the map's meaning from a legend rather than hard-coding `'#'` in our code means the same code works for any map that ships with its own legend.

**Analogy.** A dict is a phone book: look up a name, get a number. A list is a numbered queue: look up a position, get whoever stands there. A tuple is a form with a fixed number of boxes.

**Step 7: ragged rows. See the problem first, then fix it.**

In [8]:
ragged_rows = map_text.splitlines()
print([len(row_text) for row_text in ragged_rows])   # row lengths as the file has them

# Asking row 5 for column 9 fails, because that row is only 9 characters long.
try:
    print(ragged_rows[5][9])
except IndexError as problem:
    print("IndexError:", problem)

# The fix, in two moves:
#  1. rstrip() removes trailing spaces, which are noise from the exporting tool.
#  2. ljust(width) pads every row on the right with spaces up to the widest row.
trimmed_rows = [row_text.rstrip() for row_text in ragged_rows]
grid_width = max(len(row_text) for row_text in trimmed_rows)
padded_rows = [row_text.ljust(grid_width) for row_text in trimmed_rows]

print([len(row_text) for row_text in padded_rows])   # now every row is the same length

[10, 13, 10, 10, 10, 9, 9]
IndexError: string index out of range
[10, 10, 10, 10, 10, 10, 10]


**What happened.** The list of lengths shows the mess: row 1 has 13 characters (three of them trailing spaces) and rows 5 and 6 have only 9. Asking a short row for a column past its end raises an `IndexError`. After trimming noise and padding to the widest row, every row has 10 characters, so the text is now a perfect rectangle and `grid[row][column]` is safe for every row and column inside the rectangle.

A space character in the padded area means "outside the building". It is not in the legend, so later we will call it **void**.

**Step 8: find the special squares and record their addresses.**

In [9]:
# Build the grid of characters, then scan every square once.
grid = [list(row_text) for row_text in padded_rows]

markers = {}   # will map a kind such as "door" to a list of (row, column) tuples
for row_number, grid_row in enumerate(grid):                 # enumerate gives (position, item) pairs
    for column_number, character in enumerate(grid_row):
        kind = legend.get(character)                         # .get returns None if the symbol is unknown
        if kind is not None and kind not in ("wall", "floor"):
            # setdefault: if the key is missing, create it with an empty list; then hand me the list.
            markers.setdefault(kind, []).append((row_number, column_number))

print(markers)

{'start': [(1, 1)], 'door': [(3, 5), (5, 4)], 'exit': [(3, 8)]}


**What happened.** `enumerate` walks through a list and hands you each item together with its position, so we get `(row_number, grid_row)` and then `(column_number, character)` without writing `range(len(...))`. For each square whose meaning is something other than plain wall or floor, we recorded its **address**, the tuple `(row, column)`.

`markers` is the first genuinely nested value we have built: a **dict** of lists of **tuples**. The doors are at `(3, 5)` and `(5, 4)`. Check with the map: row 3 is `#.#..D.#E#`, and counting from 0 the `D` is the character at index 5. ✓

**Analogy.** A tuple like `(3, 5)` is a street address: house number and street, always exactly two parts, never edited, and usable as a name for a place. That last property matters in L02: tuples can be used as dict keys, lists cannot.

**Try it (guided experiment).** Copy your file. In the copy, replace `markers = {}` and the `setdefault` line with a `collections.defaultdict(list)`: `from collections import defaultdict`, `markers = defaultdict(list)`, and `markers[kind].append(...)`. Print the result and confirm it matches. Then return to the original file. You are comparing two real ways to say "create the list the first time I see this key". Only do this experiment after you can explain, without looking, what `setdefault` did in the line above.

**Step 9: pack Steps 5 to 8 into one function.**

In [10]:
PLAIN_KINDS = ("wall", "floor")   # kinds that do not need a recorded address


def parse_header(header_text):
    """'key: value' lines  ->  dict of str -> str."""
    header = {}
    for line in header_text.splitlines():
        key, _, value = line.partition(":")
        header[key.strip()] = value.strip()
    return header


def parse_legend(legend_text):
    """'# wall | . floor | ...'  ->  dict of symbol -> meaning."""
    legend = {}
    for entry in legend_text.split("|"):
        symbol, _, meaning = entry.strip().partition(" ")
        legend[symbol] = meaning.strip()
    return legend


def parse_floor_plan(text):
    """Whole exported file as one string  ->  one nested 'plan' dict."""
    # 1. Header and map are separated by the first blank line.
    header_text, _, map_text = text.strip("\n").partition("\n\n")
    header = parse_header(header_text)
    legend = parse_legend(header["legend"])

    # 2. Trim noise, pad to a rectangle, turn each row into a list of characters.
    trimmed_rows = [row_text.rstrip() for row_text in map_text.splitlines()]
    width = max(len(row_text) for row_text in trimmed_rows)
    grid = [list(row_text.ljust(width)) for row_text in trimmed_rows]

    # 3. Record the address of every special square, refusing symbols we do not know.
    markers = {}
    for row_number, grid_row in enumerate(grid):
        for column_number, character in enumerate(grid_row):
            if character == " ":
                continue   # padding or empty space: nothing to record
            if character not in legend:
                raise ValueError(
                    f"symbol {character!r} at row {row_number}, column {column_number} "
                    f"is not in the legend")
            kind = legend[character]
            if kind not in PLAIN_KINDS:
                markers.setdefault(kind, []).append((row_number, column_number))

    return {"header": header, "legend": legend, "grid": grid,
            "markers": markers, "height": len(grid), "width": width}


bakery_plan = parse_floor_plan(bakery_file_text)
print(bakery_plan["header"])
print(bakery_plan["markers"])
print(bakery_plan["height"], "rows x", bakery_plan["width"], "columns")

{'name': 'Corner Bakery', 'legend': '# wall | . floor | D door | S start | E exit'}
{'start': [(1, 1)], 'door': [(3, 5), (5, 4)], 'exit': [(3, 8)]}
7 rows x 10 columns


**What happened.** We took three exploratory steps and made them into one function with one input (the whole file as a string) and one output (the plan, a single nested dict). Three small choices are worth noticing:

* We raise a `ValueError` for a symbol that is not in the legend instead of silently treating it as empty. A typo in the file now produces an error that names the exact row and column, instead of a mysterious wrong answer later.
* Spaces are skipped, because they are our "outside the building" padding.
* The docstrings (the triple-quoted text under each `def`) say what goes in and what comes out in one line each.

**Step 10: reading squares safely, and asking about neighbours.**

In [11]:
def kind_at(plan, row, column):
    """What is at (row, column)? A kind name, or None if outside the map."""
    # Chained comparison: 0 <= row < height means "row is at least 0 AND below height".
    if not (0 <= row < plan["height"] and 0 <= column < plan["width"]):
        return None
    character = plan["grid"][row][column]
    return plan["legend"].get(character, "void")   # unknown symbol (the padding space) -> "void"


print(kind_at(bakery_plan, 0, 0))      # top-left corner
print(kind_at(bakery_plan, 99, 99))    # far outside the map

# A Python trap: negative indexes count from the END, so they never raise an error.
print("row -1 is quietly the LAST row:", bakery_plan["grid"][-1][:3])

# Four directions as (row_change, column_change) tuples. Rows grow downward.
DIRECTION_STEPS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
BLOCKED_KINDS = ("wall", "void")


def neighbors_of(plan, row, column):
    """Squares you can step to from (row, column): dict of direction -> (row, column)."""
    open_neighbors = {}
    for direction_name, (row_change, column_change) in DIRECTION_STEPS.items():
        neighbor_row = row + row_change
        neighbor_column = column + column_change
        kind = kind_at(plan, neighbor_row, neighbor_column)
        if kind is not None and kind not in BLOCKED_KINDS:
            open_neighbors[direction_name] = (neighbor_row, neighbor_column)
    return open_neighbors


door_row, door_column = bakery_plan["markers"]["door"][0]   # unpack the first door's address
print("first door is at", (door_row, door_column))
print(neighbors_of(bakery_plan, door_row, door_column))

wall
None
row -1 is quietly the LAST row: ['#', '#', '#']
first door is at (3, 5)
{'left': (3, 4), 'right': (3, 6)}


**What happened.** `kind_at` is the safe way to read the grid. It checks the bounds first, because Python's negative indexing would otherwise answer `row = -1` with the last row instead of an error, and a wrong answer that looks right is worse than a crash. It also turns the padding space into the kind `"void"`.

`neighbors_of` loops over a dict of four directions. The loop header `for direction_name, (row_change, column_change) in ...items()` unpacks twice: `.items()` yields `(key, value)` pairs, and the value is itself a tuple that gets unpacked into two names.

Check the result against the map: the door at `(3, 5)` has wall above and below it and floor to its left and right, so the answer `{'left': (3, 4), 'right': (3, 6)}` is right. ✓

**Reading nested data with a path.** Every pair of brackets peels off one layer:

| Path | What it means | Value |
|---|---|---|
| `plan["markers"]` | the dict of special squares | dict |
| `plan["markers"]["door"]` | the list of door addresses | list of tuples |
| `plan["markers"]["door"][1]` | the second door's address | `(5, 4)` |
| `plan["markers"]["door"][1][0]` | that address's row | `5` |

In [12]:
print(bakery_plan["markers"]["door"][1][0])   # the row of the second door

5


## 4 · Aha: every nested thing is made of three layers

All the messy structures in this series, however deep, are built from just three kinds of container:

* a **dict**: look up by name,
* a **list**: look up by position,
* a **tuple**: a small fixed-size record.

To prove it, here is a function that *reads* any nested value and describes its shape. It calls itself on each layer inside (that is called **recursion**: a function solving a problem by solving a smaller copy of the same problem).

In [13]:
def describe_shape(value):
    """Describe the nested shape of a value in one line of text."""
    if isinstance(value, dict):
        if not value:
            return "empty dict"
        key_shapes = {describe_shape(key) for key in value}
        value_shapes = {describe_shape(item) for item in value.values()}
        if len(key_shapes) == 1 and len(value_shapes) == 1:
            # Every key looks alike and every value looks alike: a lookup table.
            return f"dict of {key_shapes.pop()} -> {value_shapes.pop()}"
        # Otherwise each key plays a different role: a 'record' with named fields.
        fields = ", ".join(f"{key}: {describe_shape(item)}" for key, item in value.items())
        return "record{" + fields + "}"
    if isinstance(value, list):
        if not value:
            return "empty list"
        item_shapes = {describe_shape(item) for item in value}
        if len(item_shapes) == 1:
            return f"list of {item_shapes.pop()}"
        return "list of mixed(" + " | ".join(sorted(item_shapes)) + ")"
    if isinstance(value, tuple):
        return "(" + ", ".join(describe_shape(item) for item in value) + ")"
    return type(value).__name__


print(describe_shape(bakery_plan))

record{header: dict of str -> str, legend: dict of str -> str, grid: list of list of str, markers: dict of str -> list of (int, int), height: int, width: int}


**The aha.** The whole bakery, which looked like a wall of text five minutes ago, is *one* value whose shape fits in one line: a record with a header (dict of text to text), a legend (dict of text to text), a grid (list of list of text), markers (dict of text to a list of integer pairs), and two integers. Nothing is hidden. If you ever meet a nested value in someone else's code, you can answer "what is its shape?" with exactly this vocabulary, and you can read any part of it with a path of brackets.

**Where else this shows up.** The same move, "text in, nested value out, then read it with a path", is how programs deal with a game's tile-map files, with spreadsheet and log files (a list of records), with configuration files (a dict of dicts), and with image data (a grid of pixel values). The squares change; the three layers do not.

**Try it (guided experiment).** Copy your file. In the copy, store the grid as a **dict keyed by address** instead of a list of lists: `grid = {(row_number, column_number): character, ...}`. Rewrite `kind_at` to use it: `plan["grid"].get((row, column))` returns `None` for any address outside the map automatically. Run `describe_shape` on the new plan and compare it with the old shape. Write down one thing that got easier (hint: the bounds check) and one thing that got harder (hint: printing the map row by row). Then return to the original file. Do this only once the three-layers idea above feels solid; the experiment pays off because L02 stores whole worlds in exactly this kind of dict.

## 5 · Your turn

First a tiny checker you can reuse. It runs your function on some cases and prints **not yet** until your answer is complete and correct, then **PASS**.

In [14]:
def check(label, function, cases):
    """cases is a list of (arguments_tuple, expected_result) pairs."""
    for arguments, expected in cases:
        try:
            got = function(*arguments)
        except Exception as problem:
            print(f"{label}: not yet (raised {type(problem).__name__}: {problem})")
            return
        if got is None:
            print(f"{label}: not yet")
            return
        if got != expected:
            print(f"{label}: FAIL  arguments={arguments!r}  expected={expected!r}  got={got!r}")
            return
    print(f"{label}: PASS")


# Two tiny maps with answers you can verify by hand (the legend line is required).
wall_count_plan = parse_floor_plan("""legend: # wall | . floor | D door

#D#
#.#
###""")

corner_plan = parse_floor_plan("""legend: # wall | . floor

###
#.#
#..""")

**Challenge 1: count the squares of one kind.** Write `count_cells(plan, kind_name)` that returns how many squares of that kind the plan contains. By hand, `wall_count_plan` has 7 walls, 1 door and 1 floor.

*Hint:* two nested `for` loops over `range(plan["height"])` and `range(plan["width"])`, and `kind_at` to read each square.

**Challenge 2: count the open sides of a square.** Write `number_of_open_sides(plan, row, column)`. By hand in `corner_plan`: square `(1, 1)` has 1 open side (only downward), `(2, 1)` has 2 (up and right), and `(2, 2)` has 1.

*Hint:* you already wrote the function that lists open neighbours. A dict has a length.

**Challenge 3: a different file, the same skill.** A spreadsheet export arrives as text. Write `parse_sales_table(text)` that returns a dict from each column name to the list of that column's values, turning cells made only of digits into integers. For the text below the answer is `{'day': ['mon', 'mon', 'tue'], 'item': ['bun', 'loaf', 'bun'], 'count': [12, 4, 9]}`.

*Hint:* `split(",")` cuts a line into cells; `zip(names, cells)` pairs each name with its cell; `str.isdigit()` tells you whether a cell is all digits.

Fill in your answers by replacing `pass` below. Do not scroll to the solutions before you have tried.

In [15]:
sales_text = """day,item,count
mon,bun,12
mon,loaf,4
tue,bun,9"""


def count_cells(plan, kind_name):
    pass   # replace this line with your answer


def number_of_open_sides(plan, row, column):
    pass   # replace this line with your answer


def parse_sales_table(text):
    pass   # replace this line with your answer


def run_all_checks():
    check("count_cells", count_cells,
          [((wall_count_plan, "wall"), 7), ((wall_count_plan, "door"), 1), ((wall_count_plan, "floor"), 1)])
    check("number_of_open_sides", number_of_open_sides,
          [((corner_plan, 1, 1), 1), ((corner_plan, 2, 1), 2), ((corner_plan, 2, 2), 1)])
    check("parse_sales_table", parse_sales_table,
          [((sales_text,), {"day": ["mon", "mon", "tue"], "item": ["bun", "loaf", "bun"], "count": [12, 4, 9]})])


run_all_checks()

count_cells: not yet
number_of_open_sides: not yet
parse_sales_table: not yet


### Solutions (try first)

Each of these replaces the stub of the same name. After you define them, run `run_all_checks()` again.

In [16]:
def count_cells(plan, kind_name):
    count = 0
    for row_number in range(plan["height"]):
        for column_number in range(plan["width"]):
            if kind_at(plan, row_number, column_number) == kind_name:
                count += 1
    return count


def number_of_open_sides(plan, row, column):
    # neighbors_of already returns only the open sides, so the answer is its size.
    return len(neighbors_of(plan, row, column))


def parse_sales_table(text):
    lines = text.strip().splitlines()
    column_names = lines[0].split(",")                       # first line holds the names
    columns = {name: [] for name in column_names}            # one empty list per column
    for line in lines[1:]:
        for name, cell_text in zip(column_names, line.split(",")):
            value = int(cell_text) if cell_text.isdigit() else cell_text
            columns[name].append(value)
    return columns


run_all_checks()

count_cells: PASS
number_of_open_sides: PASS
parse_sales_table: PASS


**Why Challenge 3 matters.** Look at the answer's shape: `dict of str -> list of mixed`. A list of records and a dict of columns are two different shapes for the same table; which is better depends on what you ask of it. The lab on lookup structures (L02) is all about choosing the shape that makes your questions one-liners.

## 6 · Piece card

| Field | Entry |
|---|---|
| **Name** | `parse_floor_plan` (with helpers `kind_at`, `neighbors_of`, `describe_shape`) |
| **Input** | the exported file as one string: a header with a `legend:` line, a blank line, then the map rows |
| **Output** | one nested dict: `header`, `legend`, `grid` (list of list of str), `markers` (dict of kind to list of `(row, column)`), `height`, `width` |
| **Assumes** | rows are separated by `\n`; every non-space symbol is in the legend; blank space means "outside"; the legend entry is `symbol meaning` with a one-character symbol; no tabs, no `\r\n` line endings (**not tested here**) |
| **Three versions in this lab** | index loops (A), comprehensions with `enumerate` (B), one pass over raw characters (C) |
| **Swap rule** | any version can replace another on a valid map; all must return the same `(grid, markers)` |
| **Used later by** | L02 (Combine), L03, L05, L07, A1, A2 |

Now we save the piece as a module file so later labs can `import` it. In Jupyter the magic line `%%writefile grid_pieces.py` does this; here we use plain Python so the notebook runs anywhere. The file is also what you would type by hand into your one growing file.

In [17]:
module_source = r'''"""grid_pieces.py  (L01 piece card)

Name:        parse_floor_plan  (+ kind_at, neighbors_of, describe_shape)
Input:       exported floor-plan text: header with a 'legend:' line, a blank line, map rows
Output:      nested dict: header, legend, grid (list of list of str),
             markers (dict of kind -> list of (row, column)), height, width
Assumes:     rows split on '\n'; every non-space symbol is in the legend; blank = outside;
             one-character symbols; no tabs or '\r\n' (not tested)
Versions:    index loops / comprehensions + enumerate / single pass over characters
Swap rule:   any version may replace another; same (grid, markers) on a valid map
Used later:  L02, L03, L05, L07, A1, A2
"""

PLAIN_KINDS = ("wall", "floor")
BLOCKED_KINDS = ("wall", "void")
DIRECTION_STEPS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}


def parse_header(header_text):
    header = {}
    for line in header_text.splitlines():
        key, _, value = line.partition(":")
        header[key.strip()] = value.strip()
    return header


def parse_legend(legend_text):
    legend = {}
    for entry in legend_text.split("|"):
        symbol, _, meaning = entry.strip().partition(" ")
        legend[symbol] = meaning.strip()
    return legend


def parse_floor_plan(text):
    header_text, _, map_text = text.strip("\n").partition("\n\n")
    header = parse_header(header_text)
    legend = parse_legend(header["legend"])

    trimmed_rows = [row_text.rstrip() for row_text in map_text.splitlines()]
    width = max(len(row_text) for row_text in trimmed_rows)
    grid = [list(row_text.ljust(width)) for row_text in trimmed_rows]

    markers = {}
    for row_number, grid_row in enumerate(grid):
        for column_number, character in enumerate(grid_row):
            if character == " ":
                continue
            if character not in legend:
                raise ValueError(
                    f"symbol {character!r} at row {row_number}, column {column_number} "
                    f"is not in the legend")
            kind = legend[character]
            if kind not in PLAIN_KINDS:
                markers.setdefault(kind, []).append((row_number, column_number))

    return {"header": header, "legend": legend, "grid": grid,
            "markers": markers, "height": len(grid), "width": width}


def kind_at(plan, row, column):
    if not (0 <= row < plan["height"] and 0 <= column < plan["width"]):
        return None
    character = plan["grid"][row][column]
    return plan["legend"].get(character, "void")


def neighbors_of(plan, row, column):
    open_neighbors = {}
    for direction_name, (row_change, column_change) in DIRECTION_STEPS.items():
        neighbor_row = row + row_change
        neighbor_column = column + column_change
        kind = kind_at(plan, neighbor_row, neighbor_column)
        if kind is not None and kind not in BLOCKED_KINDS:
            open_neighbors[direction_name] = (neighbor_row, neighbor_column)
    return open_neighbors


def describe_shape(value):
    if isinstance(value, dict):
        if not value:
            return "empty dict"
        key_shapes = {describe_shape(key) for key in value}
        value_shapes = {describe_shape(item) for item in value.values()}
        if len(key_shapes) == 1 and len(value_shapes) == 1:
            return f"dict of {key_shapes.pop()} -> {value_shapes.pop()}"
        fields = ", ".join(f"{key}: {describe_shape(item)}" for key, item in value.items())
        return "record{" + fields + "}"
    if isinstance(value, list):
        if not value:
            return "empty list"
        item_shapes = {describe_shape(item) for item in value}
        if len(item_shapes) == 1:
            return f"list of {item_shapes.pop()}"
        return "list of mixed(" + " | ".join(sorted(item_shapes)) + ")"
    if isinstance(value, tuple):
        return "(" + ", ".join(describe_shape(item) for item in value) + ")"
    return type(value).__name__
'''

with open("grid_pieces.py", "w", encoding="utf-8") as module_file:
    module_file.write(module_source)

# Prove the saved module behaves exactly like the functions we built above.
import importlib
import grid_pieces
importlib.reload(grid_pieces)

print("module matches notebook:",
      grid_pieces.parse_floor_plan(bakery_file_text) == bakery_plan
      and grid_pieces.neighbors_of(bakery_plan, 3, 5) == neighbors_of(bakery_plan, 3, 5)
      and grid_pieces.describe_shape(bakery_plan) == describe_shape(bakery_plan))

module matches notebook: True


**What happened.** We wrote the source to disk, imported it, and compared its answers with the notebook's own functions on the same input. This last check is a small **contract test**: it protects you from the notebook and the module quietly drifting apart. The `r'''...'''` prefix `r` makes it a *raw* string, so sequences like `\n` inside the module source are kept as the two characters backslash and `n` rather than turned into a real line break.

## 7 · Swap test, then Combine

**The swap.** The piece "map text to grid and markers" can be written three genuinely different ways. If the contract is honest, any of them can replace any other.

* **A: index loops.** Everything done with `range(len(...))` and explicit indexes. Most explicit, and the most places to be off by one.
* **B: comprehensions and `enumerate`.** What we wrote above.
* **C: one pass over raw characters.** Never cuts the text into lines at all: it walks character by character, counting rows and columns as it goes, and starts a new row whenever it meets `\n`.

In [18]:
legend = bakery_plan["legend"]
map_text_only = bakery_file_text.strip("\n").partition("\n\n")[2]


def parse_version_a(map_text, legend):
    lines = map_text.splitlines()
    trimmed = []
    for line_index in range(len(lines)):
        trimmed.append(lines[line_index].rstrip())
    width = 0
    for line_index in range(len(trimmed)):
        if len(trimmed[line_index]) > width:
            width = len(trimmed[line_index])
    grid = []
    markers = {}
    for row_number in range(len(trimmed)):
        padded = trimmed[row_number] + " " * (width - len(trimmed[row_number]))
        grid_row = []
        for column_number in range(width):
            character = padded[column_number]
            grid_row.append(character)
            kind = legend.get(character)
            if kind is not None and kind not in ("wall", "floor"):
                if kind not in markers:
                    markers[kind] = []
                markers[kind].append((row_number, column_number))
        grid.append(grid_row)
    return grid, markers


def parse_version_b(map_text, legend):
    trimmed = [line.rstrip() for line in map_text.splitlines()]
    width = max(len(line) for line in trimmed)
    grid = [list(line.ljust(width)) for line in trimmed]
    markers = {}
    for row_number, grid_row in enumerate(grid):
        for column_number, character in enumerate(grid_row):
            kind = legend.get(character)
            if kind is not None and kind not in ("wall", "floor"):
                markers.setdefault(kind, []).append((row_number, column_number))
    return grid, markers


def parse_version_c(map_text, legend):
    grid = []
    markers = {}
    current_row = []
    row_number = 0
    for character in map_text + "\n":            # a sentinel newline closes the last row
        if character == "\n":
            grid.append(current_row)
            current_row = []
            row_number += 1
            continue
        column_number = len(current_row)
        current_row.append(character)
        kind = legend.get(character)
        if kind is not None and kind not in ("wall", "floor"):
            markers.setdefault(kind, []).append((row_number, column_number))
    for grid_row in grid:                        # same clean-up as the others, done afterwards
        while grid_row and grid_row[-1] == " ":
            grid_row.pop()
    width = max(len(grid_row) for grid_row in grid)
    for grid_row in grid:
        grid_row.extend([" "] * (width - len(grid_row)))
    return grid, markers


result_a = parse_version_a(map_text_only, legend)
result_b = parse_version_b(map_text_only, legend)
result_c = parse_version_c(map_text_only, legend)
print("A == B:", result_a == result_b)
print("B == C:", result_b == result_c)

A == B: True
B == C: True


**What happened.** Three different ways of writing the piece return exactly the same pair `(grid, markers)` on the bakery. That is the swap rule from the piece card holding up, and it is the point of writing a contract: you can change the inside without touching anything that uses the piece.

Now let us measure whether the choice makes a difference in speed. We tile the bakery into a map with 7 × 200 = 1,400 rows and time each version, keeping the best of five runs.

In [19]:
import time

big_map_text = "\n".join(map_text_only.splitlines() * 200)


def best_time_in_ms(parse_function, repeats=5):
    best_seconds = None
    for _ in range(repeats):
        start = time.perf_counter()
        parse_function(big_map_text, legend)
        elapsed = time.perf_counter() - start
        if best_seconds is None or elapsed < best_seconds:
            best_seconds = elapsed
    return best_seconds * 1000


for name, function in [("A", parse_version_a), ("B", parse_version_b), ("C", parse_version_c)]:
    print(f"{name}: {best_time_in_ms(function):.2f} ms for {len(big_map_text.splitlines())} rows")

print("all three still agree:",
      parse_version_a(big_map_text, legend) == parse_version_b(big_map_text, legend)
      == parse_version_c(big_map_text, legend))

A: 1.79 ms for 1400 rows
B: 1.60 ms for 1400 rows
C: 2.33 ms for 1400 rows
all three still agree: True


**Reading the result honestly.** The times above were measured on whatever machine ran this notebook, so yours will differ, and gaps of a few milliseconds on one run mean little. Do not memorise a ranking; do notice that three programs that look very different cost about the same order of time. What actually separates them is readability and risk, and that part is an *opinion*, not a measurement: A has the most places to make an index mistake, B reads closest to the sentence "for every square, record its address if it is special", and C avoids building the list of line strings but is the hardest to read.

**Combine.** L01 imports no earlier lab, because it is the first. The module `grid_pieces.py` is what L02 will import in its own Combine section.

**Before you go on, check yourself.** Without scrolling up, can you (1) say what the three layers of nested data are, (2) read the row of the second door from the plan with one path, and (3) explain why `kind_at` checks bounds before indexing? If any of these is shaky, redo Step 10 before starting L02. Once they are solid, the natural next move is to go beyond the lab: break the parser on purpose with a map that has an unknown symbol and read the error, or change the legend and see what still works.